# 07 - API Fundamentals

## HTTP

By the end of this module, you should be able to:
- Explain why HTTP is the foundational communication layer for machine learning inference services.
- Deconstruct an HTTP request into its core components and articulate how each part impacts model execution.
- Justify the selection of HTTP methods (`GET`, `POST`, `PUT`, `DELETE`) across different ML serving scenarios.
- Map standard HTTP status codes to specific ML production failure modes and error handling workflows.

### 1. What is HTTP in the Context of Machine Learning?

**HTTP (Hypertext Transfer Protocol)** is an application-layer network protocol that enables standardized communication between a client (mobile app, web browser, backend service) and a server (the Python inference service).

In modern MLOps, every online prediction request is an HTTP transaction. HTTP acts as the abstraction bridge: the client doesn't need to know how PyTorch, TensorFlow, or Scikit-learn works—it only needs to know how to construct a valid HTTP request.


### 2. Anatomy of an HTTP Request in ML

An HTTP request consists of three structural parts: the **Request Line**, **Headers**, and **Body**.
```
POST /v1/predict HTTP/1.1
Host: api.mlservice.com
Content-Type: application/json
Authorization: Bearer secret-token-xyz
Accept: application/json

{
"features": [5.1, 3.5, 1.4, 0.2]
}
```

- **Request Line:** Specifies the HTTP Method (`POST`), the endpoint route (`/v1/predict`), and the protocol version (`HTTP/1.1`).
- **Headers:** Key-Value metadata sent alongside the request to define context, payload format, and security parameters.
  - **Content-Type:** Tells the server how to parse the incoming payload (e.g., `application/json` for feature arrays, `multipart/form-data` for image files).
  - **Authorization:** Contains authentication credentials (e.g., `Bearer <token>`) to prevent unauthorized usage and control API access costs.
  - **Accept:** Tells the server what format the client expects back in the response (e.g., `application/json`).
- **Body:** The actual payload containing feature data required for inference.


### 3. HTTP Methods for ML APIs

Rather than memorizing dry definitions, understand why each method exists and when to use it in an ML platform.

- **GET (Retrieve Data)**
  Purpose: Used strictly to fetch resources without modifying server state.
  ML Context: Must never have a request body (by standard spec). Used for health monitoring, model metadata inspection, and system metrics.
  Examples: `/health`, `/v1/models/iris_classifier/metadata`, `/metrics`

- **POST (Submit Data / Trigger Processing)**
  Purpose: Sends data to the server to create a resource or trigger an engine computation.
  ML Context: Almost all real-time ML model inference uses POST. Inference requires complex payloads (feature matrices, nested JSONs, image files) that cannot fit inside URL parameters.
  Examples: `POST /v1/predict`, `POST /v1/embeddings`

- **PUT (Replace / Update Resource)**
  Purpose: Completely replaces an existing server-side resource with a new state.
  ML Context: Used for administrative operations such as dynamic configuration updates or reloading model weights in memory without restarting the container.
  Examples: `PUT /v1/config`

- **DELETE (Remove Resource)**
  Purpose: Removes a specified resource from the server.
  ML Context: Used to evict stale or unused model weights from memory in multi-model serving architectures.
  Examples: `DELETE /v1/models/churn_v1`


### 4. HTTP Status Codes in ML Production

Interviewers often evaluate whether you can map specific ML runtime failures to the correct HTTP status code.

### 2xx Success Series

- **200 OK**
  Meaning: Request succeeded.
  ML Context: Model completed the forward pass and returned predictions successfully.

- **201 Created**
  Meaning: Resource successfully created.
  ML Context: An asynchronous training job or batch inference job was accepted and registered on the cluster.

- **204 No Content**
  Meaning: Request succeeded, but there is no body payload in the response.
  ML Context: Health check probe passed, or a background tracking logger ping succeeded.

### 4xx Client Error Series

- **400 Bad Request**
  Meaning: Malformed request or invalid parameters.
  ML Context: Feature array has missing inputs or feature array dimensions do not match model expectations (e.g., passed 3 features when model expects 4).

- **401 Unauthorized**
  Meaning: Missing or invalid authentication token.
  ML Context: Client failed to pass an `Authorization` API key header before hitting expensive inference compute.

- **403 Forbidden**
  Meaning: Authenticated user lacks permission.
  ML Context: A tier-1 client key tries to invoke a premium LLM endpoint reserved for tier-2 enterprise accounts.

- **404 Not Found**
  Meaning: The requested URL endpoint or resource does not exist.
  ML Context: Client requests a model version or path that isn't deployed (e.g., `POST /v1/models/fraud_v99/predict`).

- **422 Validation Error**
  Meaning: Unprocessable Entity (Schema type mismatch).
  ML Context: Generated automatically by FastAPI / Pydantic when raw data types are wrong (e.g., passing a string `"five"` into a numeric field expected to be a `float`).

### 5xx Server Error Series

- **500 Internal Server Error**
  Meaning: Server encountered an unhandled exception.
  ML Context: Uncaught exception inside Python execution—such as CUDA Out-Of-Memory (OOM) error, dividing by zero in custom feature preprocessors, or missing model file paths.

### 💡 Comprehensive Mock Interview QA

- Q1: "Why wrap an ML model in an API service instead of directly executing it inside our main application backend?"

    Answer: Embedding an ML model directly inside a primary backend application tightly couples compute resources and software dependencies. Model inference requires heavy mathematical operations, specialized C++ binaries, and GPU acceleration that can degrade standard application response times. Wrapping the model inside a dedicated API microservice isolates dependencies, allows independent auto-scaling based on hardware metrics like GPU VRAM utilization, and enables cross-language consumption over standardized HTTP/REST interfaces.

- Q2: "When should an ML Engineer choose Batch Inference over Real-Time API Inference?"

    Answer: Batch inference is preferred when prediction results do not depend on immediate real-time context and can tolerate delivery latency. For example, generating product recommendations or updating credit risk scores can run on scheduled offline pipelines (e.g., via Airflow or Spark), saving pre-computed outputs to a low-latency cache like Redis. This removes runtime compute bottlenecks, lowers operational costs, and insulates the system from real-time model failure.

- Q3: "Why shouldn't we use a GET request with query parameters to send input feature vectors for ML model predictions?"

    Answer: First, HTTP spec conventions dictate that GET requests are intended for retrieving state without side effects, and many proxy tools discard body data inside GET requests. Second, query parameters have strict URL length limits (typically around 2,048 characters), making it impossible to pass high-dimensional feature vectors, image arrays, or text documents. POST requests allow structured JSON payloads without size constraints.

- Q4: "In a production ML API, what is the operational difference between a 422 Validation Error and a 500 Internal Server Error?"

    Answer: A 422 Unprocessable Entity is a client-side input error handled gracefully by schema validation engines like Pydantic before reaching model code (e.g., passing string values to numeric fields). A 500 Internal Server Error indicates an uncaught server-side runtime failure—such as GPU Out-Of-Memory, missing dependencies, or unhandled null values during feature preprocessing. 422 errors point to client bugs, whereas 500 errors signal system or model service instability that needs engineer intervention.

## FastAPI

By the end of this module, you should be able to:
- Explain the role of a web framework and distinguish between the framework layer (FastAPI) and the underlying ASGI web server (`Uvicorn`).
- Compare **Flask**, **Django**, and **FastAPI** across performance, async capabilities, and validation features.
- Define routing mechanisms and map incoming HTTP URLs to underlying Python path operation functions.
- Build production-ready FastAPI endpoints leveraging type hints, Pydantic validation, and dependency injection.

### 1. What is a Web Framework?

Python by itself cannot listen to TCP ports or parse incoming HTTP wire protocols. A **web framework** acts as an intermediary application layer that sits between the low-level HTTP server and your high-level machine learning code.

```
[ Incoming Network Traffic ]
│
▼
┌────────────────────────────────────────┐
│ HTTP Web Server (Uvicorn / Starlette) │  <-- Opens sockets, parses HTTP protocol bytes
└──────────────────┬─────────────────────┘
│ ASGI Interface
▼
┌────────────────────────────────────────┐
│ Web Framework (FastAPI)                │  <-- Matches routes, validates schemas, runs middleware
└──────────────────┬─────────────────────┘
│ Processed Python Objects
▼
┌────────────────────────────────────────┐
│ ML Code / Model Execution (.predict)   │  <-- Executes feature preprocessing & forward pass
└
```

- **Python Runtime:** Executes code, loads PyTorch/Scikit-learn libraries, and manages memory.
- **HTTP Server (Uvicorn):** The low-level ASGI engine that listens on network ports (e.g., `8000`), accepts TCP connections, and converts raw HTTP wire bytes into structured ASGI scope dictionaries.
- **Web Framework (FastAPI):** The application framework that inspects request paths, routes traffic to specific functions, enforces data schemas via Pydantic, and formats output objects into HTTP responses.


### 2. Why FastAPI? (Framework Comparison)

In modern ML engineering, FastAPI has largely replaced legacy frameworks like Flask and Django for building inference services.

| Feature / Dimension | Flask | Django | FastAPI |
| :--- | :--- | :--- | :--- |
| **Server Interface** | WSGI (Synchronous / Blocking) | WSGI / Async hybrid | Native ASGI (Asynchronous) |
| **Data Validation** | Manual parsing (`request.get_json()`) | Form/ORM serialisers | Automatic via **Pydantic** & **Type Hints** |
| **API Documentation** | Requires manual third-party plugins | Requires manual third-party plugins | **Automatic OpenAPI / Swagger UI** |
| **Execution Architecture** | Thread-per-request | Heavy monolithic stack | Event loop / Non-blocking async |
| **Primary Use Case** | Legacy microservices, simple web apps | Full-stack monolithic web applications | High-performance ML APIs & Microservices |

Core FastAPI Advantages for ML

- **Type Safety & Type Hints:** Leverages standard Python 3.10+ type annotations (`int`, `str`, `List[float]`) to parse and validate incoming data structures.
- **Automatic Data Validation:** Intercepts invalid input payloads (e.g., missing features or wrong data types) before they hit expensive model code, instantly returning a `422 Unprocessable Entity` response.
- **Native Async Support:** Built on Starlette's ASGI event loop, allowing long-running I/O operations (like fetching embeddings or querying databases) to run concurrently without blocking main server threads.
- **Auto-Generated Interactive Docs:** Instantly generates `/docs` (Swagger UI) and `/redoc` interfaces from route definitions, simplifying frontend and platform integration.


### 3. Understanding Routing

**Routing** is the mechanism of binding a specific incoming HTTP request URL path and method to a corresponding Python execution function (called a **Path Operation Function**).

```
[ Incoming Request ]
GET  /               ────────►  root()           ──► Returns {"message": "Service Online"}
POST /v1/predict     ────────►  predict()        ──► Runs Model Forward Pass
GET  /v1/metadata    ────────►  get_metadata()   ──► Returns Model Version Info
```

- Decorator (@app.post(...)): Registers the function below it into FastAPI's internal routing table for a specific URL pattern.

- Path Operation Function (def predict_endpoint): The Python function executed whenever a request matches the defined method and path.

In [1]:
### Route Anatomy in FastAPI
from fastapi import FastAPI

app = FastAPI()

# Decorator specifies: HTTP Method (POST) + Path Route (/v1/predict)
@app.post("/v1/predict")
def predict_endpoint(payload: dict):
    # Path Operation Function: Executed when POST /v1/predict is invoked
    return {"status": "success"}

### 4. Production FastAPI ML Implementation
Below is a production-grade FastAPI application featuring health probes, route metadata, schema enforcement, and error handling.

In [ ]:
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel, Field
from typing import List
import time

# 1. Instantiate Core FastAPI App
app = FastAPI(
    title="Iris Classifier API",
    description="Production-grade model serving endpoint using FastAPI and Pydantic",
    version="1.0.0"
)

# 2. Define Request Schema using Pydantic
class InferenceRequest(BaseModel):
    features: List[float] = Field(
        ..., 
        description="4-element feature vector: [sepal length, sepal width, petal length, petal width]",
        example=[5.1, 3.5, 1.4, 0.2]
    )

# 3. Define Response Schema
class InferenceResponse(BaseModel):
    prediction: int
    class_name: str
    confidence: float
    latency_ms: float

# 4. Root & Health Check Routes (GET)
@app.get("/", status_code=status.HTTP_200_OK)
def root():
    return {"message": "ML Serving API is running."}

@app.get("/health", status_code=status.HTTP_200_OK)
def health_check():
    # In production, check if model weights are loaded in memory
    return {"status": "healthy", "model_loaded": True}

# 5. Model Inference Route (POST)
@app.post(
    "/v1/predict", 
    response_model=InferenceResponse,
    status_code=status.HTTP_200_OK
)
def predict(payload: InferenceRequest):
    # Validate feature dimensions
    if len(payload.features) != 4:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail=f"Expected 4 features, but received {len(payload.features)}."
        )

    start_time = time.time()

    try:
        # Mocking model inference execution
        mock_prediction = 0
        mock_class = "setosa"
        mock_confidence = 0.98

        latency = (time.time() - start_time) * 1000

        return InferenceResponse(
            prediction=mock_prediction,
            class_name=mock_class,
            confidence=mock_confidence,
            latency_ms=round(latency, 2)
        )
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"Internal model execution error: {str(e)}"
        )

### 💡 Comprehensive Mock Interview QA

- Q1: "What is the difference between WSGI and ASGI, and why is ASGI important for ML model serving?"

    Answer: WSGI (Web Server Gateway Interface) is a synchronous specification used by legacy frameworks like Flask. In WSGI, each incoming request blocks a server worker thread until execution completes. ASGI (Asynchronous Server Gateway Interface) is the modern asynchronous standard used by FastAPI/Uvicorn. ASGI enables a single server process to handle thousands of concurrent I/O-bound requests using an event loop. For ML APIs, ASGI allows the server to continue accepting incoming network requests without blocking while long-running inference tasks run asynchronously in worker pools.

- Q2: "How does FastAPI use Pydantic and Python type hints to reduce boilerplate validation code?"

    Answer: FastAPI inspects path operation function signatures and Pydantic type annotations at startup. When an HTTP request arrives, FastAPI automatically parses the raw JSON body, checks field existence, validates data types (e.g., ensuring numeric strings like "5.1" are cast to floats), and maps the payload into a strongly-typed Pydantic object. If validation fails, FastAPI automatically generates a structured 422 Unprocessable Entity response detailing the exact field and validation error before your function code runs.

- Q3: "In FastAPI, what is the difference between declaring a path operation function as def predict() versus async def predict() when executing heavy CPU/GPU ML inference?"

    Answer: Declaring an endpoint as async def tells FastAPI that the function contains non-blocking I/O operations (like await aiohttp...). If you put heavy, synchronous CPU/GPU inference code (like PyTorch tensor computations) directly inside an async def function, it will block the entire main ASGI event loop, stalling all other concurrent requests. For heavy CPU/GPU compute, you should either declare the endpoint as a standard def predict()—which instructs FastAPI to run the function inside an external thread pool—or offload computation to a dedicated worker queue (like Triton or Celery).

### FastAPI vs Flask vs Django, quickly

| | Flask | Django | FastAPI |
|---|---|---|---|
| Validation | manual | forms / DRF serializers | automatic, from type hints (Pydantic) |
| Async support | limited | growing | native |
| Auto-generated docs | needs an extension | needs DRF | built in, at /docs |
| Best fit | small, simple services | full web apps with ORM/admin | APIs, especially ML/data services |

Automatic validation and auto-generated OpenAPI docs are the two reasons FastAPI is the default choice for model-serving APIs today -- both come directly from the type hints and Pydantic models already used elsewhere in this chapter, not extra work.

Install what the rest of this notebook needs, if not already present:

In [9]:
import importlib.util, subprocess, sys

for pkg in ["fastapi", "uvicorn", "pydantic", "requests"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import fastapi, pydantic, uvicorn
print("fastapi", fastapi.__version__, "| pydantic", pydantic.__version__, "| uvicorn", uvicorn.__version__)

fastapi 0.141.1 | pydantic 2.13.4 | uvicorn 0.52.0



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: pip install --upgrade pip
